In [1]:
import pandas as pd
import numpy as np

file_path = "smart_factory_dataset.xlsx"
df = pd.read_excel(file_path)
df.head()


,Machine_ID,Machine_Mode,RPM,Vibration_mm_s,Pressure_bar,Temperature_C,Defective
0,MCH-001,Standard,1631.7,6.848,4.46,70.0,0
1,MCH-002,High-Speed,2766.9,11.463,7.83,116.8,1
2,MCH-003,High-Speed,2439.5,7.383,6.92,91.7,1
3,MCH-004,Standard,1382.4,3.017,5.79,58.7,0
4,MCH-005,Eco,963.9,3.409,4.06,47.5,0


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Machine_Mode_enc"] = le.fit_transform(df["Machine_Mode"])

X = df[["RPM", "Vibration_mm_s", "Pressure_bar", "Machine_Mode_enc"]]
y = df["Temperature_C"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Coefficients:")
for feat, coef in zip(X.columns, model.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"\nIntercept: {model.intercept_:.4f}")
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

Coefficients:
  RPM: 0.0148
  Vibration_mm_s: 3.5422
  Pressure_bar: 2.6487
  Machine_Mode_enc: 0.6441

Intercept: 10.4443
R² Score:  0.9419
RMSE:      3.8985


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

X = df[["RPM"]]
y = df["Temperature_C"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Slope (RPM coef): {model.coef_[0]:.4f}")
print(f"Intercept:        {model.intercept_:.4f}")
print(f"R² Score:         {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:             {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

Slope (RPM coef): 0.0312
Intercept:        17.9862
R² Score:         0.7769
RMSE:             7.6405


In [6]:
import pickle

with open("linear_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved → linear_model.pkl")

Model saved → linear_model.pkl


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df[["RPM", "Vibration_mm_s", "Pressure_bar", "Temperature_C", "Machine_Mode"]]
y = df["Defective"]

preprocessor = ColumnTransformer(transformers=[
    ("ohe", OneHotEncoder(), ["Machine_Mode"])
], remainder="passthrough")

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9000

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95        88
           1       0.75      0.25      0.38        12

    accuracy                           0.90       100
   macro avg       0.83      0.62      0.66       100
weighted avg       0.89      0.90      0.88       100

